# Kimi K3 → свой GDN-2/MLA гибрид: JAX-версия для TPU v5e-8

Это порт прошлого PyTorch-ноутбука на **JAX/Flax/Optax** под одну TPU-машину v5e-8 (8 чипов, по 16 GB HBM, вместе 128 GB, один хост).

**Что изменилось по сравнению с PyTorch-версией и почему:**

| Тема | Было | Стало |
|---|---|---|
| Фреймворк | PyTorch | JAX + Flax + Optax (тот же стек, что для обучения на TPU) |
| Работа с Hub | `torch.frombuffer` | `numpy` + `ml_dtypes` (bf16/fp8 без torch), загрузка идёт в RAM хоста |
| Dequant MXFP4 | torch на CPU | `jit`-функция на чипах + numpy-версия для сверки |
| Ridge-статистики | float64 на CPU | накопление `AtA`, `AtY` в **float32 с `Precision.HIGHEST` на TPU**, решение системы в float64 на хосте (TPU не умеет f64 аппаратно) |
| Параллелизм | нет | батч шардится по оси `fsdp` (8 чипов), `AtA` собирается all-reduce'ом автоматически (GSPMD) |
| Healing | `nn.Module` + `.requires_grad` | Flax + разделение параметров на trainable/frozen, FSDP-шардинг параметров и состояния оптимизатора |
| Отладка без TPU | нет | `EMULATE_CPU_DEVICES=8` эмулирует 8 устройств на CPU |

**Что проверено, а что нет.** Этот ноутбук я выполнил целиком на **8 эмулированных CPU-устройствах**: шардинг, all-reduce, jit и Flax-обучение отработали корректно. Настоящего TPU у меня нет, поэтому **скорость, потребление HBM и поведение `bf16`-математики на v5e я не мерил**. Твой предыдущий прогон PyTorch-версии, судя по всему, считался на CPU хоста (если ты не подключал `torch_xla`), так что первый реальный прогон на чипах будет именно с этим ноутбуком.

**Что по-прежнему неизвестно про K3** (см. чек-лист в конце): имена тензоров, `d_model`, `d_latent`, `I`, число слоёв, формат MXFP4, формула SiTU, линейность латентных проекций.

In [1]:
import os
# >0 -> emulate N CPU devices (debug without a TPU). On the TPU VM leave it at 0 / unset.
EMULATE_CPU_DEVICES = int(os.environ.get("EMULATE_CPU_DEVICES", "0"))
if EMULATE_CPU_DEVICES:
    os.environ["XLA_FLAGS"] = f"--xla_force_host_platform_device_count={EMULATE_CPU_DEVICES}"
    os.environ["JAX_PLATFORMS"] = "cpu"

import json, re, struct, math, time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from functools import partial

import numpy as np
import ml_dtypes
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, NamedSharding, PartitionSpec as P

REPO = "moonshotai/Kimi-K3"
REV = "main"
RUN_REMOTE = False          # True -> talk to the Hub (runs on the TPU host CPU, downloads into host RAM/disk)
OUT_DIR = "k3_subset"
os.makedirs(OUT_DIR, exist_ok=True)

# persistent compile cache: TPU compiles are slow, this makes re-runs fast
try:
    jax.config.update("jax_compilation_cache_dir", "/tmp/jax_cache")
except Exception as e:
    print("compile cache not enabled:", e)

HIGHEST = jax.lax.Precision.HIGHEST      # use for anything where fp32 fidelity matters (stats, folds, eval)

devices = jax.devices()
ON_TPU = devices[0].platform == "tpu"
mesh = Mesh(np.array(devices).reshape(-1), ("fsdp",))      # one axis: data-parallel batch AND FSDP param sharding
N_DEV = mesh.size
REPL = NamedSharding(mesh, P())                              # replicated
BATCH2 = NamedSharding(mesh, P("fsdp", None))                # [batch, feat] sharded on batch

print("jax", jax.__version__, "| platform:", devices[0].platform, "| devices:", N_DEV,
      "| kind:", getattr(devices[0], "device_kind", "?"))
if not ON_TPU and not EMULATE_CPU_DEVICES:
    print("WARNING: not running on TPU. Set EMULATE_CPU_DEVICES=8 to test sharding on CPU.")

jax 0.11.2 | platform: cpu | devices: 8 | kind: cpu


## Часть 1. Выгрузить только нужную часть с Hub (хост, numpy)

**Теория.** Веса в шардах `*.safetensors`; индекс `model.safetensors.index.json` даёт «тензор → шард»; в начале шарда 8 байт длины заголовка и JSON с `dtype`, `shape`, `data_offsets`. Хаб отдаёт HTTP `Range`, поэтому качаем заголовок и только нужные байты.

**Про TPU.** Сеть и парсинг работают на CPU TPU-VM, тензоры лежат в RAM хоста (у v5e-8 хост-память намного больше, чем HBM). На чипы данные уходят позже, уже после отбора. Если сервер игнорирует `Range` и отдаёт `200`, мы падаем с ошибкой, чтобы случайно не скачать весь шард.

In [2]:
import requests

def hf_url(path, repo=REPO, rev=REV):
    return f"https://huggingface.co/{repo}/resolve/{rev}/{path}"

def get_hf_token():
    tok = os.environ.get("HF_TOKEN")
    if tok:
        return tok
    try:
        from huggingface_hub import get_token
        return get_token()
    except Exception:
        return None

_TOKEN = get_hf_token()
HEADERS = {"Authorization": f"Bearer {_TOKEN}"} if _TOKEN else {}

def http_get(url, byte_range=None, retries=4):
    headers = dict(HEADERS)
    if byte_range:
        headers["Range"] = f"bytes={byte_range[0]}-{byte_range[1]}"
    last = None
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, allow_redirects=True, timeout=180)
        except requests.RequestException as e:
            last = e; time.sleep(2 ** attempt); continue
        if byte_range and r.status_code == 200:
            raise RuntimeError("Server ignored Range and returned the whole file; aborting.")
        if r.status_code in (200, 206):
            return r.content
        if r.status_code in (429, 500, 502, 503, 504):
            last = RuntimeError(f"HTTP {r.status_code}"); time.sleep(2 ** attempt); continue
        r.raise_for_status()
    raise RuntimeError(f"GET failed for {url}: {last}")

NP_DTYPES = {
    "BF16": ml_dtypes.bfloat16, "F16": np.float16, "F32": np.float32,
    "U8": np.uint8, "I8": np.int8, "I32": np.int32, "I64": np.int64,
    "F8_E4M3": ml_dtypes.float8_e4m3fn, "F8_E5M2": ml_dtypes.float8_e5m2,
    "F8_E8M0": getattr(ml_dtypes, "float8_e8m0fnu", None),
}
DTYPE_BYTES = {"BF16": 2, "F16": 2, "F32": 4, "U8": 1, "I8": 1, "I32": 4, "I64": 8,
               "F8_E4M3": 1, "F8_E5M2": 1, "F8_E8M0": 1}

def load_index(repo=REPO, rev=REV):
    return json.loads(http_get(hf_url("model.safetensors.index.json", repo, rev)))["weight_map"]

_header_cache = {}
def read_shard_header(shard, repo=REPO, rev=REV):
    key = (repo, rev, shard)
    if key in _header_cache:
        return _header_cache[key]
    url = hf_url(shard, repo, rev)
    (n,) = struct.unpack("<Q", http_get(url, (0, 7)))
    header = json.loads(http_get(url, (8, 7 + n)))
    header.pop("__metadata__", None)
    _header_cache[key] = (header, 8 + n)
    return _header_cache[key]

def summarize_names(weight_map, max_rows=80):
    pats = defaultdict(int)
    for k in weight_map:
        pats[re.sub(r"\.(\d+)(?=\.|$)", ".{N}", k)] += 1
    for k, c in sorted(pats.items())[:max_rows]:
        print(f"{c:6d}  {k}")
    print(f"... {len(pats)} distinct patterns, {len(weight_map)} tensors total")

def plan_fetch(names, weight_map, repo=REPO, rev=REV):
    by_shard = defaultdict(list)
    for n in names:
        by_shard[weight_map[n]].append(n)
    plan = []
    for shard, ns in by_shard.items():
        header, base = read_shard_header(shard, repo, rev)
        url = hf_url(shard, repo, rev)
        for n in ns:
            info = header[n]
            s, e = info["data_offsets"]
            plan.append(dict(name=n, url=url, start=base + s, end=base + e - 1,
                             dtype=info["dtype"], shape=info["shape"]))
    return plan

def _to_array(buf, dtype_str, shape):
    dt = NP_DTYPES[dtype_str]
    if dt is None:                                            # dtype unknown to ml_dtypes -> raw bytes
        return np.frombuffer(bytearray(buf), dtype=np.uint8).copy()
    return np.frombuffer(bytearray(buf), dtype=dt).reshape(shape).copy()

def fetch_tensors(names, weight_map, workers=8, dry_run=True, repo=REPO, rev=REV):
    plan = plan_fetch(names, weight_map, repo, rev)
    total = sum(p["end"] - p["start"] + 1 for p in plan)
    print(f"{len(plan)} tensors, {total / 1e9:.2f} GB")
    if dry_run:
        return None
    def get(p):
        return p["name"], _to_array(http_get(p["url"], (p["start"], p["end"])), p["dtype"], p["shape"])
    with ThreadPoolExecutor(workers) as ex:
        return dict(ex.map(get, plan))

def fetch_tensor_slice(name, expert_ids, weight_map, repo=REPO, rev=REV):
    """FUSED experts stored as one [E, ...] tensor: read only rows `expert_ids` along dim 0."""
    shard = weight_map[name]
    header, base = read_shard_header(shard, repo, rev)
    info = header[name]
    shape = info["shape"]
    per = math.prod(shape[1:]) * DTYPE_BYTES[info["dtype"]]
    start0 = base + info["data_offsets"][0]
    url = hf_url(shard, repo, rev)
    def get(e):
        s = start0 + e * per
        return _to_array(http_get(url, (s, s + per - 1)), info["dtype"], shape[1:])
    with ThreadPoolExecutor(4) as ex:
        return np.stack(list(ex.map(get, expert_ids)), 0)

In [3]:
# --- Step 1a: inspect naming scheme (needs network) ---
if RUN_REMOTE:
    weight_map = load_index()
    summarize_names(weight_map)
else:
    print("RUN_REMOTE=False: skipping. Set it to True to list tensor-name patterns of the repo.")

RUN_REMOTE=False: skipping. Set it to True to list tensor-name patterns of the repo.


In [4]:
# --- Step 1b: choose what to fetch. ADJUST the regexes after looking at the listing above. ---
FFN_RE    = re.compile(r"^model\.layers\.(\d+)\.mlp\.")      # <- placeholder
EXPERT_RE = re.compile(r"\.experts\.(\d+)\.")                 # <- per-expert tensors (non-fused layout)

LAYERS = range(0, 8)
KEEP_EXPERTS = set(range(0, 16))          # replace with a usage-based selection

def want(name):
    m = FFN_RE.match(name)
    if not m or int(m.group(1)) not in LAYERS:
        return False
    e = EXPERT_RE.search(name)
    return e is None or int(e.group(1)) in KEEP_EXPERTS

if RUN_REMOTE:
    names = [n for n in weight_map if want(n)]
    fetch_tensors(names, weight_map, dry_run=True)            # ALWAYS look at the size first
    # tensors = fetch_tensors(names, weight_map, dry_run=False)   # dict name -> numpy (host RAM)
else:
    print("skipped (offline)")

skipped (offline)


## Часть 2. Dequant MXFP4 → bf16 (на чипах)

**Теория.** MXFP4: значения FP4 E2M1 (16 кодов: знак + 3 бита; величины `0, .5, 1, 1.5, 2, 3, 4, 6`), общий масштаб E8M0 (`2^(e-127)`) на блок из 32 элементов вдоль оси свёртки. В файле два `uint8`-тензора: упакованные значения `[..., K/2]` и scales `[..., K/32]`.

**Про TPU.** v5e не имеет нативного fp8, поэтому веса в MXFP4/E8M0 на чип отправляем как **сырые `uint8`**, а декодируем внутри `jit`: gather по таблице из 16 значений и умножение на масштаб. Результат bf16 остаётся на чипах. Если scales пришли как `float8_e8m0`, на хосте берём их `uint8`-биты (`as_u8`).

**Не проверено на реальном K3:** порядок nibble (здесь low-first) и layout. Сверь результат с эталонным dequant из репозитория K3 на одном тензоре.

In [5]:
FP4_LUT_NP = np.array([0, .5, 1, 1.5, 2, 3, 4, 6, -0., -.5, -1, -1.5, -2, -3, -4, -6], dtype=np.float32)
MX_BLOCK = 32

def as_u8(x):
    """Raw bits as uint8 (handles float8_e8m0 scales that came from ml_dtypes)."""
    x = np.asarray(x)
    return x if x.dtype == np.uint8 else x.view(np.uint8)

def dequant_mxfp4_np(packed, scales, dtype=ml_dtypes.bfloat16, low_first=True):
    """Host reference implementation (numpy)."""
    packed, scales = as_u8(packed), as_u8(scales)
    lo, hi = (packed & 0x0F).astype(np.int64), (packed >> 4).astype(np.int64)
    a, b = (lo, hi) if low_first else (hi, lo)
    vals = np.stack([FP4_LUT_NP[a], FP4_LUT_NP[b]], -1).reshape(*packed.shape[:-1], packed.shape[-1] * 2)
    scale = np.repeat(np.exp2(scales.astype(np.float32) - 127.0), MX_BLOCK, axis=-1)
    return (vals * scale).astype(dtype)

@partial(jax.jit, static_argnames=("out_dtype", "low_first"))
def dequant_mxfp4(packed, scales_u8, out_dtype=jnp.bfloat16, low_first=True):
    """Device version. packed: uint8 [..., K/2]; scales_u8: uint8 [..., K/32] -> [..., K]."""
    lut = jnp.asarray(FP4_LUT_NP)
    lo = (packed & 0x0F).astype(jnp.int32)
    hi = (packed >> 4).astype(jnp.int32)
    a, b = (lo, hi) if low_first else (hi, lo)
    vals = jnp.stack([lut[a], lut[b]], axis=-1).reshape(*packed.shape[:-1], packed.shape[-1] * 2)
    scale = jnp.repeat(jnp.exp2(scales_u8.astype(jnp.float32) - 127.0), MX_BLOCK, axis=-1)
    return (vals * scale).astype(out_dtype)

def quantize_mxfp4_np(W):
    """Reference quantizer for the self-test only."""
    *lead, K = W.shape
    assert K % MX_BLOCK == 0
    Wb = W.astype(np.float32).reshape(*lead, K // MX_BLOCK, MX_BLOCK)
    amax = np.maximum(np.abs(Wb).max(-1, keepdims=True), 1e-30)
    exp = np.ceil(np.log2(amax / 6.0))
    x = Wb / np.exp2(exp)
    idx = np.abs(np.abs(x)[..., None] - FP4_LUT_NP[:8]).argmin(-1)
    code_ = (idx + (x < 0) * 8).reshape(*lead, K)
    packed = (code_[..., 0::2] | (code_[..., 1::2] << 4)).astype(np.uint8)
    scales = np.clip(exp[..., 0] + 127, 0, 254).astype(np.uint8)
    return packed, scales

# --- self-test: numpy vs device, and layout sensitivity ---
rng = np.random.default_rng(0)
W = (rng.standard_normal((64, 256)) * 0.02).astype(np.float32)
q, s = quantize_mxfp4_np(W)
W_np = dequant_mxfp4_np(q, s, dtype=np.float32)
W_dev = np.asarray(dequant_mxfp4(jnp.asarray(q), jnp.asarray(s), out_dtype=jnp.float32))
rel = np.linalg.norm(W - W_np) / np.linalg.norm(W)
print(f"packed {q.shape} scales {s.shape} | quantization rel. error {rel:.3f}")
print("numpy vs device max abs diff:", float(np.abs(W_np - W_dev).max()))
assert rel < 0.25 and np.allclose(W_np, W_dev), "dequant is broken"
W_bad = dequant_mxfp4_np(q, s, dtype=np.float32, low_first=False)
print(f"wrong nibble order error: {np.linalg.norm(W - W_bad) / np.linalg.norm(W):.3f}")

packed (64, 128) scales (64, 8) | quantization rel. error 0.119
numpy vs device max abs diff: 0.0
wrong nibble order error: 1.422


## Часть 3. Слайсинг ширины residual stream (`d_model` → `d_student`)

**Теория.** Residual stream общий для всех слоёв, поэтому набор каналов `idx` **один** для embedding, lm_head, всех RMSNorm и всех проекций, читающих из потока (`in`) или пишущих в него (`out`). Важность считаем по весам (teacher не запустить): канал важен, если его сильно читают (`||W[:, j]||` с учётом `γ_j`) и он несёт энергию в эмбеддингах.

После отрезания меняется RMS: `γ_s = γ_t · sqrt(ρ · d_t / d_s)`, где `ρ` доля сохранённой энергии. Это грубая поправка, healing всё равно нужен.

**Если можно, оставь `d_model` студента равным `d_model` K3** для тех блоков, куда переносишь FFN: тогда слайсинг не нужен.

**Про TPU.** Редукции по осям считаются на чипах; индексация по `idx` (gather) дешёвая, но выравнивай `d_student` по 128 (размер MXU у v5e), иначе часть чипа простаивает.

In [6]:
def channel_importance(read, d_model, embed_key, norm_readers):
    """read: callable(key) -> array (dequantized). norm_readers: [(gamma_key, [matrix keys [out, d_model]])]."""
    imp = jnp.zeros((d_model,), jnp.float32)
    E = jnp.asarray(read(embed_key), jnp.float32)
    imp += jnp.mean(E ** 2, axis=0)
    for gamma_key, mats in norm_readers:
        g2 = jnp.asarray(read(gamma_key), jnp.float32) ** 2
        for k in mats:
            W = jnp.asarray(read(k), jnp.float32)
            imp += g2 * jnp.sum(W ** 2, axis=0) / W.shape[0]
    return imp

def pick_channels(imp, d_student):
    idx = jnp.sort(jax.lax.top_k(imp, d_student)[1])
    rho = float(imp[idx].sum() / imp.sum())
    return idx, rho

def slice_tensor(W, idx, role):
    if role == "in":    return jnp.take(W, idx, axis=-1)     # W [out, d]
    if role == "out":   return jnp.take(W, idx, axis=0)      # W [d, in]
    if role == "vec":   return jnp.take(W, idx, axis=0)      # RMSNorm gamma
    if role == "embed": return jnp.take(W, idx, axis=1)      # [V, d]
    raise ValueError(role)

def rescale_gamma(gamma_s, rho, d_t, d_s):
    return gamma_s * math.sqrt(rho * d_t / d_s)

def check_mxu_alignment(**dims):
    """v5e MXU is 128x128: dims that are not multiples of 128 waste compute."""
    bad = {k: v for k, v in dims.items() if v % 128}
    print("MXU alignment:", "OK" if not bad else f"NOT multiples of 128 -> {bad}")

# --- synthetic demo with heavy-tailed channel scales (like real LLM outlier channels) ---
d_t, d_s, V = 256, 128, 1000
k0 = jax.random.split(jax.random.PRNGKey(0), 4)
chan_scale = jnp.exp(jax.random.normal(k0[0], (d_t,)))
fake = {"emb": jax.random.normal(k0[1], (V, d_t)) * chan_scale, "g0": jnp.ones(d_t),
        "q0": jax.random.normal(k0[2], (64, d_t)), "k0": jax.random.normal(k0[3], (64, d_t))}
imp = channel_importance(lambda k: fake[k], d_t, "emb", [("g0", ["q0", "k0"])])
idx, rho = pick_channels(imp, d_s)
print(f"kept {idx.shape[0]}/{d_t} channels, energy share rho = {rho:.3f}")
print("emb ->", slice_tensor(fake['emb'], idx, 'embed').shape, "| q0 ->", slice_tensor(fake['q0'], idx, 'in').shape,
      "| gamma factor:", round(math.sqrt(rho * d_t / d_s), 3))
check_mxu_alignment(d_student=d_s)

kept 128/256 channels, energy share rho = 0.881
emb -> (1000, 128) | q0 -> (64, 128) | gamma factor: 1.327
MXU alignment: OK


## Часть 4. Склейка экспертов в dense FFN

**Теория.** MoE считает `y = Σ_i w_i · E_i(x)` по выбранным экспертам. Склейка `k` экспертов по intermediate-оси даёт dense FFN, считающий `Σ_i E_i(x)` **для каждого токена**: лишние эксперты дают шум, веса `w_i` потеряны. Исправления: вшить в `down_proj` средний routing weight каждого эксперта; если латентные проекции **линейны**, вшить их в матрицы (`d_model → d_latent → эксперты → d_latent → d_model` превращается в FFN прямо в `d_model`).

**Про TPU.** Вшивание это большие матричные произведения: делаем их в float32 с `Precision.HIGHEST` (дефолт для f32 на TPU это bf16-проходы, для складывания весов это слишком грубо) и только потом приводим к bf16.

Toy Latent-MoE использует SiLU вместо SiTU (формулу K3 я не знаю).

In [7]:
def init_toy_moe(key, d=64, dl=32, I=24, E=32):
    k = jax.random.split(key, 6)
    n = jax.random.normal
    return dict(W_in=n(k[0], (dl, d)) / d ** .5, W_out=n(k[1], (d, dl)) / dl ** .5,
                router=n(k[2], (E, dl)) / dl ** .5 * 3,
                gate=n(k[3], (E, I, dl)) / dl ** .5, up=n(k[4], (E, I, dl)) / dl ** .5,
                down=n(k[5], (E, dl, I)) / I ** .5)

@partial(jax.jit, static_argnames=("topk",))
def moe_forward(p, x, topk=2):
    E = p["router"].shape[0]
    xl = jnp.dot(x, p["W_in"].T, precision=HIGHEST)                                  # [N, dl]
    logits = jnp.dot(xl, p["router"].T, precision=HIGHEST)
    vals, ids = jax.lax.top_k(logits, topk)
    w = jax.nn.softmax(vals, axis=-1)
    route = (jax.nn.one_hot(ids, E) * w[..., None]).sum(1)                           # [N, E]
    h = jax.nn.silu(jnp.einsum("nl,eil->nei", xl, p["gate"], precision=HIGHEST)) * \
        jnp.einsum("nl,eil->nei", xl, p["up"], precision=HIGHEST)
    y_e = jnp.einsum("nei,eli->nel", h, p["down"], precision=HIGHEST)
    y = jnp.dot((route[..., None] * y_e).sum(1), p["W_out"].T, precision=HIGHEST)
    return y, route

def make_clustered_inputs(N, d, n_clusters=16, noise=0.4, seed=0):
    """Cluster centers fixed (same for train/test); `seed` only changes the samples."""
    centers = np.random.default_rng(123).standard_normal((n_clusters, d)).astype(np.float32)
    rng = np.random.default_rng(seed)
    ids = rng.integers(0, n_clusters, N)
    return centers[ids] + noise * rng.standard_normal((N, d)).astype(np.float32)

def merge_experts_to_dense(experts, gate_w=None, W_in=None, W_out=None):
    """experts: list of {'gate': [I, d_l], 'up': [I, d_l], 'down': [d_l, I]}.
    gate_w: [k] expected routing weight per expert (None -> weight 1). W_in [d_l, d_model], W_out [d_model, d_l]."""
    f32 = lambda a: jnp.asarray(a, jnp.float32)
    G = jnp.concatenate([f32(e["gate"]) for e in experts], 0)
    U = jnp.concatenate([f32(e["up"]) for e in experts], 0)
    gw = jnp.ones(len(experts)) if gate_w is None else f32(gate_w)
    D = jnp.concatenate([f32(e["down"]) * gw[i] for i, e in enumerate(experts)], 1)
    if W_in is not None:
        G, U = jnp.dot(G, f32(W_in), precision=HIGHEST), jnp.dot(U, f32(W_in), precision=HIGHEST)
    if W_out is not None:
        D = jnp.dot(f32(W_out), D, precision=HIGHEST)
    return {"G": G, "U": U, "D": D}

@jax.jit
def dense_hidden(m, x):
    return jax.nn.silu(jnp.dot(x, m["G"].T, precision=HIGHEST)) * jnp.dot(x, m["U"].T, precision=HIGHEST)

@jax.jit
def dense_forward(m, x):
    return jnp.dot(dense_hidden(m, x), m["D"].T, precision=HIGHEST)

def moe_experts_as_list(p, ids):
    return [{"gate": p["gate"][e], "up": p["up"][e], "down": p["down"][e]} for e in ids]

@jax.jit
def cos_and_relmse(pred, target):
    cos = jnp.mean(jnp.sum(pred * target, -1) / (jnp.linalg.norm(pred, axis=-1) * jnp.linalg.norm(target, axis=-1) + 1e-9))
    rel = jnp.sum((pred - target) ** 2) / jnp.sum(target ** 2)
    return cos, rel

## Часть 5. Ridge-refit `down_proj` на 8 чипах

**Теория.** После склейки заново подгоняем `down_proj`, чтобы dense-слой воспроизводил выход MoE-слоя на реальных активациях: `D^T = (AᵀA + λI)⁻¹ AᵀY`, где `A = act(Gx)·(Ux)`, `Y` выход teacher-слоя. Обучения нет, нужна одна матрица `kI × kI` на слой.

**Как это ложится на TPU v5e-8:**
- калибровочный батч шардится по оси `fsdp` (`P("fsdp", None)`); `Aᵀ A` считается на каждом чипе по своему куску, а суммирование по чипам (all-reduce) XLA вставляет сам, потому что результат объявлен реплицированным;
- накопление `AtA`, `AtY` идёт **в float32 на устройстве** с `Precision.HIGHEST` (донорим буферы, чтобы не копировать);
- итоговая система решается **на хосте в float64** (TPU не умеет f64 аппаратно). Для `kI` порядка десятка тысяч это ~1-2 GB на `AtA` и десятки секунд на слой;
- для реального слоя `Y` берётся у teacher, а не из toy-функции (см. ниже).

Метрики считаем на **отложенной** выборке.

In [8]:
@partial(jax.jit, donate_argnums=(0, 1))
def accumulate_stats(AtA, AtY, A, Y):
    A, Y = A.astype(jnp.float32), Y.astype(jnp.float32)
    return (AtA + jnp.dot(A.T, A, precision=HIGHEST),
            AtY + jnp.dot(A.T, Y, precision=HIGHEST))

def solve_down(AtA_dev, AtY_dev, lam=1e-3):
    """Solve in float64 on the host. Returns new down_proj [d_out, kI] as float32 numpy."""
    AtA = np.asarray(AtA_dev, dtype=np.float64)
    AtY = np.asarray(AtY_dev, dtype=np.float64)
    reg = lam * np.mean(np.diag(AtA)) * np.eye(AtA.shape[0])
    return np.linalg.solve(AtA + reg, AtY).T.astype(np.float32)

def shard_batch(x):
    return jax.device_put(x, BATCH2)

d, dl, I, E, topk, k = 64, 32, 24, 32, 2, 8
moe = init_toy_moe(jax.random.PRNGKey(0), d, dl, I, E)
x_tr = shard_batch(make_clustered_inputs(40000, d, seed=1))       # 40000 % 8 == 0
x_te = shard_batch(make_clustered_inputs(8000,  d, seed=2))
y_tr, route_tr = moe_forward(moe, x_tr, topk=topk)
y_te, _ = moe_forward(moe, x_te, topk=topk)
print("x_tr sharding:", x_tr.sharding.spec, "| devices holding data:", len(x_tr.sharding.device_set))

usage = np.asarray(route_tr.mean(0))                                # mean routing weight per expert
top_used = sorted(np.argsort(-usage)[:k].tolist())
rand_ids = sorted(np.random.default_rng(0).permutation(E)[:k].tolist())
print("usage-selected experts:", top_used, "| routing mass covered:", round(float(usage[top_used].sum() / usage.sum()), 3))

results = {}
def evaluate(name, m):
    c, r = cos_and_relmse(dense_forward(m, x_te), y_te)
    results[name] = (float(c), float(r))

Wi, Wo = moe["W_in"], moe["W_out"]
evaluate("1 random + concat",  merge_experts_to_dense(moe_experts_as_list(moe, rand_ids), None, Wi, Wo))
m2 = merge_experts_to_dense(moe_experts_as_list(moe, top_used), None, Wi, Wo);            evaluate("2 usage + concat", m2)
m3 = merge_experts_to_dense(moe_experts_as_list(moe, top_used), usage[top_used], Wi, Wo); evaluate("3 usage + weights", m3)

kI = k * I
AtA = jax.device_put(jnp.zeros((kI, kI), jnp.float32), REPL)
AtY = jax.device_put(jnp.zeros((kI, d), jnp.float32), REPL)
for a, b in zip(jnp.split(x_tr, 10), jnp.split(y_tr, 10)):                                 # 4000 rows each, 4000 % 8 == 0
    a = jax.device_put(a, BATCH2); b = jax.device_put(b, BATCH2)
    AtA, AtY = accumulate_stats(AtA, AtY, dense_hidden(m3, a), b)
m4 = dict(m3); m4["D"] = jnp.asarray(solve_down(AtA, AtY, lam=1e-3))
evaluate("4 usage + weights + ridge", m4)

print(f"\n{'variant':32s} {'cosine':>8s} {'rel.MSE':>9s}")
for name, (c, r) in results.items():
    print(f"{name:32s} {c:8.3f} {r:9.3f}")
assert results["4 usage + weights + ridge"][1] < results["3 usage + weights"][1], "ridge should reduce held-out error"
check_mxu_alignment(kI=kI, d_model=d)     # toy dims are small; on real dims aim for multiples of 128

x_tr sharding: P('fsdp', None) | devices holding data: 8
usage-selected experts: [2, 3, 4, 9, 14, 21, 24, 31] | routing mass covered: 0.601



variant                            cosine   rel.MSE
1 random + concat                   0.101    21.258
2 usage + concat                    0.181    15.867
3 usage + weights                   0.167     1.004
4 usage + weights + ridge           0.680     0.453
MXU alignment: NOT multiples of 128 -> {'kI': 192, 'd_model': 64}


**Как читать результат.** Это toy на случайных весах, на K3 числа будут другими и переносить выводы нельзя. Видна механика: голая склейка завышает амплитуду (`rel.MSE` больше 10: суммируются все `k` экспертов с весом 1), routing weights возвращают масштаб, ridge заметно поднимает `cosine`. Повтори четыре варианта на 4-8 реальных слоях и для нескольких `k` (например 4, 8, 16).

**Откуда `Y` и `A` для реального teacher.** Нужны входы и выходы MoE-слоя на калибровочных токенах. K3 целиком не запустить (даже 128 GB HBM этой машины для него на порядки мало), поэтому: (а) меньший teacher с тем же типом слоёв, например Kimi Linear 48B-A3B (проверь по карточке); он в bf16 около 96 GB, так что на v5e-8 влезает только с квантованием или с послойной загрузкой; (б) послойный стриминг слоёв K3 с диска на небольшом наборе токенов; для ridge хватает десятков тысяч токенов на слой.

**Альтернатива склейке.** Оставить маленький MoE: `M` лучших экспертов, строки роутера срезать до них. Специализация сохраняется. Сравни оба варианта по `cosine` и `rel.MSE` на нескольких слоях.

## Часть 6. Бюджет: параметры студента и память v5e-8

**Теория.** Dense FFN из `k` экспертов (3 матрицы) весит `3 · k · I · d`. Общий бюджет: `embeddings + Σ_layers (attention + FFN)`. Числа ниже это **плейсхолдеры**, подставь свои из `config.json`.

**Память при обучении.** На v5e 16 GB HBM на чип. Грубая оценка на параметр: замороженный вес в bf16 = 2 байта; обучаемый = bf16 копия (2) + fp32 master (4) + Adam `m`,`v` (8) = 14 байт. Всё делится на 8 при FSDP-шардинге, плюс активации (их снижает `remat`). Это оценка порядка величины, а не измерение.

In [9]:
def plan_budget(target_params, n_layers, d_model, I, attn_params_per_layer, vocab, tie_embeddings=False, n_mats=3):
    emb = vocab * d_model * (1 if tie_embeddings else 2)
    ffn_budget = target_params - emb - n_layers * attn_params_per_layer
    if ffn_budget <= 0:
        raise ValueError("Budget is smaller than embeddings + attention; reduce layers/d_model.")
    k_exact = ffn_budget / (n_layers * n_mats * I * d_model)
    k = max(1, int(k_exact))
    return k, k_exact, emb + n_layers * (attn_params_per_layer + n_mats * k * I * d_model)

def estimate_train_hbm_gb(n_params, trainable_frac, n_dev=8, hbm_per_chip_gb=16):
    frozen = n_params * (1 - trainable_frac) * 2
    train = n_params * trainable_frac * 14
    per_chip = (frozen + train) / n_dev / 1e9
    return per_chip, per_chip / hbm_per_chip_gb

# ---- PLACEHOLDER numbers (NOT K3's real config) ----
cfg_guess = dict(n_layers=24, d_model=2048, I=1024, attn_params_per_layer=25e6, vocab=160_000)
for target in (2e9, 3e9, 5e9):
    try:
        k_, kx, tot = plan_budget(target, **cfg_guess)
        s1, f1 = estimate_train_hbm_gb(tot, trainable_frac=0.3, n_dev=N_DEV)
        s2, f2 = estimate_train_hbm_gb(tot, trainable_frac=1.0, n_dev=N_DEV)
        print(f"target {target/1e9:.0f}B -> k={k_} (exact {kx:.2f}), total {tot/1e9:.2f}B | "
              f"HBM/chip w/o activations: stage1 (30% trainable) {s1:.1f} GB ({f1:.0%}), stage2 (all) {s2:.1f} GB ({f2:.0%})")
    except ValueError as e:
        print(f"target {target/1e9:.0f}B -> {e}")

target 2B -> k=4 (exact 4.93), total 1.86B | HBM/chip w/o activations: stage1 (30% trainable) 1.3 GB (8%), stage2 (all) 3.3 GB (20%)
target 3B -> k=11 (exact 11.55), total 2.92B | HBM/chip w/o activations: stage1 (30% trainable) 2.0 GB (13%), stage2 (all) 5.1 GB (32%)
target 5B -> k=24 (exact 24.80), total 4.88B | HBM/chip w/o activations: stage1 (30% trainable) 3.4 GB (21%), stage2 (all) 8.5 GB (53%)


## Часть 7. Healing на TPU: Flax + Optax + FSDP

**Теория.** Перенесённые веса это хорошая инициализация, но mixer-слои (GDN-2 вместо KDA), residual (нет AttnRes) и нормы не совпадают с теми, к которым «привыкли» FFN. Стадии:
1. FFN, embeddings, lm_head заморожены; учатся mixer, нормы, аналог AttnRes; loss = hidden-MSE + KL; LR порядка `1e-3…3e-4`.
2. Всё разморожено, LR порядка `1e-5…3e-5`, больше веса на KL.
3. Обычный pretrain/SFT.

**Как это устроено под v5e-8:**
- параметры делятся на `trainable` и `frozen` **до** дифференцирования: градиенты и состояние Adam есть только у trainable, память на стадии 1 заметно меньше;
- FSDP: каждый тензор шардится по самой большой оси, делящейся на 8 (`fsdp_spec`); состояние оптимизатора наследует шардинг через `jit(opt.init)`; батч шардится по `fsdp`;
- `donate_argnums` отдаёт буферы под новые значения (нет двойной памяти на параметры);
- для настоящей модели добавь `jax.checkpoint` (remat) на блоки и bf16-вычисления; frozen-часть можно хранить в bf16 (`frozen_dtype`).

**Про teacher.** Полный K3 как online-teacher недоступен: используй меньшего teacher или офлайн-данные (генерации K3 через API с top-k logprobs, тогда loss `sparse_kd_loss`).

In [10]:
import flax.linen as nn
import optax
from flax.traverse_util import flatten_dict, unflatten_dict

# ---------------- losses ----------------
def kd_loss(s_logits, t_logits, T=1.0):
    s_lp = jax.nn.log_softmax(s_logits.astype(jnp.float32) / T, -1)
    t_p = jax.nn.softmax(t_logits.astype(jnp.float32) / T, -1)
    t_lp = jnp.log(t_p + 1e-20)
    return jnp.mean(jnp.sum(t_p * (t_lp - s_lp), -1)) * (T * T)

def sparse_kd_loss(s_logits, topk_ids, topk_logprobs):
    """KD vs an API teacher that only returns top-k logprobs. Both sides renormalized on the top-k support."""
    s_lp = jnp.take_along_axis(jax.nn.log_softmax(s_logits.astype(jnp.float32), -1), topk_ids, axis=-1)
    s_lp = s_lp - jax.nn.logsumexp(s_lp, axis=-1, keepdims=True)
    t_p = jax.nn.softmax(topk_logprobs.astype(jnp.float32), -1)
    return -jnp.mean(jnp.sum(t_p * s_lp, -1))

def hidden_mse(s_hiddens, t_hiddens, projs=None):
    """Per-block MSE normalized by teacher variance. projs: optional list of [d_s, d_t] matrices (or None)."""
    tot = 0.0
    for i, (s, t) in enumerate(zip(s_hiddens, t_hiddens)):
        s, t = s.astype(jnp.float32), t.astype(jnp.float32)
        if projs is not None and projs[i] is not None:
            s = s @ projs[i]
        tot = tot + jnp.mean((s - t) ** 2) / (jnp.var(t) + 1e-6)
    return tot / len(s_hiddens)

# ---------------- parameter partitioning + FSDP ----------------
def partition_params(params, frozen_markers, frozen_dtype=None):
    """Split a nested param dict into (trainable, frozen) flat dicts by substring match on the '/'-joined path."""
    flat = flatten_dict(params, sep="/")
    fr = {k: v for k, v in flat.items() if any(m in k for m in frozen_markers)}
    tr = {k: v for k, v in flat.items() if k not in fr}
    if frozen_dtype is not None:
        fr = {k: v.astype(frozen_dtype) for k, v in fr.items()}
    return tr, fr

def merge_params(tr, fr):
    return unflatten_dict({**tr, **fr}, sep="/")

def fsdp_spec(shape, n_dev, axis="fsdp"):
    """Shard the largest axis divisible by n_dev; replicate otherwise (biases, tiny tensors)."""
    cands = [(s, i) for i, s in enumerate(shape) if s >= n_dev and s % n_dev == 0]
    if not cands:
        return P()
    _, i = max(cands)
    spec = [None] * len(shape); spec[i] = axis
    return P(*spec)

def shard_tree(tree, mesh):
    return jax.tree.map(lambda x: jax.device_put(x, NamedSharding(mesh, fsdp_spec(x.shape, mesh.size))), tree)

# ---------------- train step ----------------
def make_train_step(apply_fn, opt, a=1.0, b=0.1, T=1.0):
    def loss_fn(tr, fr, ids, t_logits, t_hiddens):
        s_logits, s_hiddens = apply_fn({"params": merge_params(tr, fr)}, ids)
        return a * kd_loss(s_logits, t_logits, T) + b * hidden_mse(s_hiddens, t_hiddens)

    @partial(jax.jit, donate_argnums=(0, 1))
    def step(tr, opt_state, fr, ids, t_logits, t_hiddens):
        loss, grads = jax.value_and_grad(loss_fn)(tr, fr, ids, t_logits, t_hiddens)
        updates, opt_state = opt.update(grads, opt_state, tr)
        return optax.apply_updates(tr, updates), opt_state, loss
    return step

# ---------------- smoke test: tiny model, 8-way FSDP ----------------
class TinyBlock(nn.Module):
    d: int
    @nn.compact
    def __call__(self, x):
        x = x + nn.Dense(self.d, name="mixer")(x)                          # stands for GDN-2/MLA (trainable, stage 1)
        h = nn.silu(nn.Dense(2 * self.d, name="ffn_up")(x))                # transferred FFN (frozen, stage 1)
        return x + nn.Dense(self.d, name="ffn_down")(h)

class TinyStudent(nn.Module):
    V: int = 64
    d: int = 64
    L: int = 3
    @nn.compact
    def __call__(self, ids):
        h, hs = nn.Embed(self.V, self.d, name="embed")(ids), []
        for i in range(self.L):
            h = TinyBlock(self.d, name=f"blocks_{i}")(h); hs.append(h)
        return nn.Dense(self.V, use_bias=False, name="lm_head")(h), hs

FROZEN_MARKERS = ("ffn", "embed", "lm_head")
model = TinyStudent()
ids = jax.random.randint(jax.random.PRNGKey(1), (8, 16), 0, 64)                     # batch 8 % 8 devices == 0
student_params = model.init(jax.random.PRNGKey(2), ids)["params"]
leaves, treedef = jax.tree.flatten(student_params)
tkeys = jax.random.split(jax.random.PRNGKey(3), len(leaves))
teacher_params = jax.tree.unflatten(treedef, [p + 0.05 * jax.random.normal(kk, p.shape) for p, kk in zip(leaves, tkeys)])
t_logits, t_hiddens = jax.jit(model.apply)({"params": teacher_params}, ids)

tr, fr = partition_params(student_params, FROZEN_MARKERS)
tr, fr = shard_tree(tr, mesh), shard_tree(fr, mesh)
tr_before = {k: np.asarray(v) for k, v in tr.items()}
fr_before = {k: np.asarray(v) for k, v in fr.items()}
print(f"trainable: {len(tr)} tensors | frozen: {len(fr)} tensors | example sharding:",
      {k: str(v.sharding.spec) for k, v in list(tr.items())[:2]})

opt = optax.chain(optax.clip_by_global_norm(1.0), optax.adamw(1e-3))
opt_state = jax.jit(opt.init)(tr)
step = make_train_step(model.apply, opt)

ids_s = jax.device_put(ids, BATCH2)
tl_s = jax.device_put(t_logits, NamedSharding(mesh, P("fsdp", None, None)))
th_s = [jax.device_put(h, NamedSharding(mesh, P("fsdp", None, None))) for h in t_hiddens]

losses = []
for _ in range(60):
    tr, opt_state, loss = step(tr, opt_state, fr, ids_s, tl_s, th_s)
    losses.append(float(loss))
print(f"stage 1 loss: {losses[0]:.4f} -> {losses[-1]:.4f}")
assert losses[-1] < losses[0]
for k_, v_ in fr.items():
    assert np.array_equal(fr_before[k_], np.asarray(v_)), f"frozen param changed: {k_}"
assert all(not np.array_equal(tr_before[k_], np.asarray(v_)) for k_, v_ in tr.items() if "kernel" in k_ or "bias" in k_), "some trainable param did not update"
print("frozen params unchanged, trainable params updated: OK")

K = 5
s_logits = jax.random.normal(jax.random.PRNGKey(4), (2, 4, 50))
t_ids = jax.random.randint(jax.random.PRNGKey(5), (2, 4, K), 0, 50)
t_lp = jax.nn.log_softmax(jax.random.normal(jax.random.PRNGKey(6), (2, 4, K)), -1)
print("sparse_kd_loss:", round(float(sparse_kd_loss(s_logits, t_ids, t_lp)), 4))

trainable: 6 tensors | frozen: 14 tensors | example sharding: {'blocks_0/mixer/bias': "P('fsdp',)", 'blocks_0/mixer/kernel': "P(None, 'fsdp')"}


stage 1 loss: 0.2733 -> 0.0531
frozen params unchanged, trainable params updated: OK


sparse_kd_loss: 1.8311


## Часть 8. Сборка state dict студента и сохранение

Шаблон: собираешь словарь с ключами **своей** архитектуры (имена ниже условные). Для реальных весов сначала части 1-2 (получить и декодировать), затем 3-5 для каждого слоя. Сохранение идёт через `safetensors.flax` (умеет bf16), с хоста, а не через `pickle`.

In [11]:
def build_student_ffn_from_moe(moe_tensors, layer, kept_ids, gate_w, W_in=None, W_out=None,
                               down_override=None, prefix="blocks"):
    """moe_tensors: {'gate': [E, I, d_l], 'up': [E, I, d_l], 'down': [E, d_l, I]} for ONE layer.
    Returns {student_key: bf16 array}. Names are placeholders."""
    experts = [{"gate": moe_tensors["gate"][e], "up": moe_tensors["up"][e], "down": moe_tensors["down"][e]}
               for e in kept_ids]
    m = merge_experts_to_dense(experts, gate_w, W_in, W_out)
    if down_override is not None:                       # ridge-refit result
        m["D"] = jnp.asarray(down_override)
    bf = lambda a: jnp.asarray(a).astype(jnp.bfloat16)
    return {f"{prefix}.{layer}.ffn.gate_proj.weight": bf(m["G"]),
            f"{prefix}.{layer}.ffn.up_proj.weight":   bf(m["U"]),
            f"{prefix}.{layer}.ffn.down_proj.weight": bf(m["D"])}

student_sd = build_student_ffn_from_moe(
    {"gate": moe["gate"], "up": moe["up"], "down": moe["down"]},
    layer=0, kept_ids=top_used, gate_w=usage[top_used], W_in=moe["W_in"], W_out=moe["W_out"], down_override=m4["D"])
for k_, v_ in student_sd.items():
    print(k_, tuple(v_.shape), v_.dtype)

try:
    from safetensors.flax import save_file, load_file
    path = f"{OUT_DIR}/student_ffn_demo.safetensors"
    save_file(dict(student_sd), path)
    back = load_file(path)
    assert all(np.array_equal(np.asarray(student_sd[k_].astype(jnp.float32)), np.asarray(back[k_].astype(jnp.float32))) for k_ in student_sd)
    print("saved + reloaded OK:", path)
except ImportError:
    print("pip install safetensors to save")

blocks.0.ffn.gate_proj.weight (192, 64) bfloat16
blocks.0.ffn.up_proj.weight (192, 64) bfloat16
blocks.0.ffn.down_proj.weight (64, 192) bfloat16
saved + reloaded OK: k3_subset/student_ffn_demo.safetensors


## Чек-лист перед запуском на реальном K3 и v5e-8

1. Запусти первую ячейку и убедись, что `platform: tpu`, `devices: 8`. Если написано `cpu`, JAX не увидел TPU (проверь установку `jax[tpu]` и что нет другого процесса, занявшего чипы).
2. Выведи `summarize_names(weight_map)` и подгони `FFN_RE`, `EXPERT_RE`; выясни, fused ли эксперты; найди тензоры scales для MXFP4.
3. Прогони `dequant_mxfp4` на одном реальном тензоре и сравни с эталонным dequant из репозитория K3 (порядок nibble, layout scales).
4. Выпиши `d_model`, `d_latent`, `I`, число блоков; проверь, линейны ли латентные проекции и нет ли между ними нормы.
5. Реализуй SiTU в студенте (здесь SiLU только для toy).
6. Реши, нужен ли слайсинг ширины (часть 3) или `d_model` студента равен `d_model` K3; выровняй размеры по 128.
7. Получи калибровочные `A` и `Y` (меньший teacher или послойный стриминг); выбери `k` через часть 6.
8. Проверь `cosine` и `rel.MSE` на отложенных токенах для вариантов из части 5 на нескольких слоях, прежде чем собирать всю модель.
9. На реальной модели добавь `jax.checkpoint`, замерь HBM (`jax.devices()[0].memory_stats()`) и throughput; мои оценки из части 6 это порядок величины.
10. Проверь условия Kimi K3 License для производных моделей.